In [16]:
from pathlib import Path
import os

try:
    ROOT = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if p.name == "rat_pose")
except StopIteration:
    pass
os.chdir(ROOT)
print(os.getcwd())

/mnt/c/Users/jiewang/OneDrive - University of Texas Medical Branch/code/rat_pose


In [17]:
import deeplabcut
import os
import yaml
from deeplabcut.modelzoo import build_weight_init
import shutil
from modules.train_utils import resolve_best_transfer_paths, delete_created_training_artifacts
from modules.dlc_utils import set_transform_prob,fix_dlc_config

project_path = 'projects/rat'
config_path = os.path.join(project_path, "config.yaml")
fix_dlc_config(config_path)
project_config = deeplabcut.auxiliaryfunctions.read_config(config_path)

In [18]:
shuffle = 5
pretrained_shuffle = 3

In [19]:
# superanimal_name = 'superanimal_mouse'
superanimal_name = 'superanimal_quadruped'
model_name = "rtmpose_s"
# detector_name="fasterrcnn_resnet50_fpn_v2"
detector_name = None
weight_init = build_weight_init(
            cfg = config_path,
            super_animal= superanimal_name,
            model_name=model_name,
            detector_name=detector_name,
            with_decoder=False
)

In [20]:
import numpy as np
np.random.seed(42)
delete_created_training_artifacts(project_path, shuffle=shuffle, iteration=0)
dt = deeplabcut.create_training_dataset(
    config_path, 
    Shuffles=[shuffle],    
    weight_init=weight_init, 
    net_type=model_name,  
    detector_type=detector_name,
    userfeedback=False)

snapshot_path, detector_path = resolve_best_transfer_paths(project_path, source_shuffle=pretrained_shuffle)

Removed 2 dataset entries, 1 metadata entries, and 1 model directories for shuffle5.


INFO:root:Default augmenter albumentations not available for engine Engine.PYTORCH: using albumentations instead


Using snapshot_path: projects/rat/dlc-models-pytorch/iteration-0/ratOct2-trainset95shuffle3/train/snapshot-best-310.pt
Using detector_path: projects/rat/dlc-models-pytorch/iteration-0/ratOct2-trainset95shuffle3/train/snapshot-detector-best-170.pt


# augmentation

In [21]:
from deeplabcut.core.config import read_config_as_dict
import deeplabcut.pose_estimation_pytorch as dlc_torch

loader = dlc_torch.DLCLoader(
    config=config_path,  
    trainset_index=0,
    shuffle=shuffle,
)

# Get the pytorch config
pytorch_config_path = loader.model_folder / "pytorch_config.yaml"
model_cfg = read_config_as_dict(pytorch_config_path)

model_cfg["detector"]["model"]["freeze_bn_stats"] = False
model_cfg["data"]["train"]['covering'] = True
model_cfg["data"]["train"]['grayscale'] = True
model_cfg["data"]["train"]['gaussian_noise'] = 20

# model_cfg["data"]["train"]['crop_sampling'] = {'height': 512, 'width': 512, 'max_shift': 0.2, 'method': 'hybrid'}

model_cfg["data"]["train"]["hflip"] = {
    "p": 0.5,  # 50% probability
    "symmetries": [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
}

dlc_torch.config.write_config(pytorch_config_path, model_cfg)

In [ ]:
deeplabcut.train_network(
    config_path,
    shuffle=shuffle,
    epochs=400,
    save_epochs=10,
    detector_epochs=200,
    superanimal_name=superanimal_name,
    batch_size=16,
    keepdeconvweights=False,
    device="cuda:0",
    superanimal_transfer_learning=True,
    # snapshot_path=snapshot_path,
    # detector_path=detector_path,  # if top-down
)


Training with configuration:
data:
  bbox_margin: 20
  colormode: RGB
  inference:
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
  train:
    affine:
      p: 0.5
      rotation: 30
      scaling: [1.0, 1.0]
      translation: 0
    gaussian_noise: 20
    motion_blur: True
    normalize_images: True
    top_down_crop:
      width: 256
      height: 256
    random_bbox_transform:
      shift_factor: 0.16
      shift_prob: 0.3
      scale_factor: [0.75, 1.25]
      scale_prob: 1.0
      p: 1.0
    covering: True
    grayscale: True
    hflip:
      p: 0.5
      symmetries: [[9, 15], [10, 11], [12, 14], [13, 18], [16, 17]]
detector:
  data:
    colormode: RGB
    inference:
      normalize_images: True
    train:
      affine:
        p: 0.5
        rotation: 30
        scaling: [1.0, 1.0]
        translation: 40
      collate:
        type: ResizeFromDataSizeCollate
        min_scale: 0.4
        max_scale: 1.0
        min_short_side: 128
        max_sh